# Destination-choice + stay/move model via xlogit (long format)

xlogit port of `us/modeling_mnl.ipynb`. Every person gets `alt=0` (stay) plus one row per move
alternative in the file (`alt=1..total_move_alternatives`). Movers always have their true chosen
destination at `alt=1` by construction (same convention as the biogeme notebook: "for movers, they
all choose alternative 1 by construction; all stayers choose alternative 0"), so `choice` is a
direct function of `alt` and `STAY` — no PUMA/CHOSEN matching needed.

Stay and move don't share a formula (different variables entirely — origin-side `.ORIG` columns
for stay, `ALT{i}_<suffix>` destination columns for move), so the move block is reshaped with
xlogit's `wide_to_long`, and the stay block (one row per person, no alternative-varying columns)
is built directly.


In [ ]:
import numpy as np
import pandas as pd
from xlogit import MultinomialLogit
from xlogit.utils import wide_to_long

In [ ]:
year = 2018
total_move_alternatives = 50

In [ ]:
ALT_VARYING_SUFFIXES = [
    "TOT_POP",
    "DIST",
    "CBSA",
    "STATE",
    "TYPE",
    "HH_MED_INC",
    "POVERTY_PCT",
    "YR_SINCE_MED_STRUCTURE",
    "MED_HOUSE_VALUE_OVER_MED_HH_INC",
    "MED_RENT_PCT_HH_INC",
    "MED_OWNER_COST_HH_INC_PCT",
    "UNEMP_RATE",
    "HOUSE_VACANCY_PCT",
    "COLLEGE_PCT",
    "FOREIGN_BORN_PCT",
    "OWN_AGE_PCT",
    "OWN_RACE_PCT",
    "OWN_GROUP_PCT",
    "HH_WITH_CHILD_PCT",
    "ENT_JOBS_PCT",
    "MED_TRAVEL_TIME",
    "ALT_COMMUTE_PCT",
    "MIL_PCT",
]

# NOTE: intentionally still a fixed column list, not the whole file - reading all ~2,200
# non-ALT columns (mostly unused census fields) would reintroduce the RAM blowup this
# conversation spent a while fixing, even with the .iloc[0:500] slice below (parquet reads
# every row for whatever columns you ask for, then the slice throws most of them away).
INDIV_COLS = [
    "STAY",
    "CHILD",
    "IN_COLLEGE",
    "FOREIGN",
    "BLACK",
    "AAPI",
    "INDIAN",
    "LATINO",
    "IN_MILITARY",
    "NAICS_GOVT",
    "NAICS_GOODS_TRADE",
    "NAICS_LICENSE",
    "NAICS_HIGH_ED",
    "NAICS_AGR_EXT",
    "AGE_18_34",
    "AGE_35_64",
    "AGE_OVER_65",
    "ST",
    "POBP",
    "NAME_NUM.ORIG",
    "TYPE_NUM.ORIG",
    "AGE_18_22",
    "AGE_23_29",
    "AGE_30_39",
    "AGE_40_49",
    "AGE_50_64",
    "CHILD_UNDER_6",
    "CHILD_6_TO_17",
    "MARRIED_MORE_THAN_YEAR",
    "RECENTLY_MARRIED",
    "RECENTLY_WIDOWED_OR_DIVORCED",
    "WORK2_MAR",
    "WORK1_MAR",
    "SINGLE_PARENT",
    "EDU_BACHELORS",
    "EDU_HIGH",
    "EDU_NOHIGH",
    "Proportion of people 18-34.ORIG",
    "Proportion of people 35-64.ORIG",
    "Proportion of people 65+.ORIG",
    "Proportion of households with children.ORIG",
    "Proportion of people in college.ORIG",
    "Proportion of people Black.ORIG",
    "Proportion of people AAPI.ORIG",
    "Proportion of people Indian.ORIG",
    "Proportion of people Latino.ORIG",
    "Proportion foreign born.ORIG",
    "Proportion of people in military.ORIG",
    "NAICS_GROUP_PCT_GOVT.ORIG",
    "NAICS_GROUP_PCT_GOODS_TRADE.ORIG",
    "NAICS_GROUP_PCT_LICENSE.ORIG",
    "NAICS_GROUP_PCT_HIGH_ED.ORIG",
    "NAICS_GROUP_PCT_AGR_EXT.ORIG",
    "Proportion of people struggling.ORIG",
    "Years since median structure built.ORIG",
    "Median house value over median household income.ORIG",
    "Median gross rent as a percentage of household income.ORIG",
    "Median selected monthly owner costs as percentage of household income.ORIG",
    "Unemployment rate.ORIG",
    "House vacancy proportion.ORIG",
    "Median travel time.ORIG",
    "Proportion alternative commute.ORIG",
    (
        "Median Household Income (In 2018 Inflation Adjusted Dollars)."
        "Median Household Income (In 2018 Inflation Adjusted Dollars).SE_A14006_001.ORIG"
    ),
]
needed_cols = INDIV_COLS + [
    f"ALT{i}_{suf}"
    for i in range(1, total_move_alternatives + 1)
    for suf in ALT_VARYING_SUFFIXES
]
needed_cols = list(dict.fromkeys(needed_cols))

df = pd.read_parquet(f"../data/us_estdata_{year}.parquet", columns=needed_cols)

cat_cols = df.dtypes[df.dtypes == "category"].keys()
df[cat_cols] = df[cat_cols].apply(lambda x: x.astype(int))
obj_cols = df.dtypes[df.dtypes == "object"].keys()
df[obj_cols] = df[obj_cols].apply(lambda x: x.astype(int))

df["person_id"] = np.arange(len(df))
print(df.shape)


### Move block: `wide_to_long` + trivial choice

In [ ]:
move_long = wide_to_long(
    df,
    id_col="person_id",
    alt_list=[f"ALT{i}" for i in range(1, total_move_alternatives + 1)],
    alt_name="alt_label",
    varying=ALT_VARYING_SUFFIXES,
    sep="_",
    alt_is_prefix=True,
)
move_long["alt"] = move_long["alt_label"].str[len("ALT") :].astype(int)
# for movers, alt=1 is always their chosen destination by construction (see modeling_mnl.ipynb)
move_long["choice"] = ((move_long["alt"] == 1) & (move_long["STAY"] == 0)).astype(int)

same_state = (move_long["ST"] == move_long["STATE"]).astype(float)
same_cbsa = (move_long["NAME_NUM.ORIG"] == move_long["CBSA"]).astype(float)
same_type_t34 = (move_long["TYPE_NUM.ORIG"] == 0).astype(float)
same_type_metro = (move_long["TYPE_NUM.ORIG"] == 1).astype(float)
same_type_nonmetro = (move_long["TYPE_NUM.ORIG"] == 2).astype(float)
alt_type_t34 = (move_long["TYPE"] == 0).astype(float)
alt_type_metro = (move_long["TYPE"] == 1).astype(float)
alt_type_nonmetro = (move_long["TYPE"] == 2).astype(float)

move_long["log_tot_pop"] = np.log(move_long["TOT_POP"])
move_long["destchoice_dist"] = (1 - same_cbsa) * move_long["DIST"]
move_long["destchoice_logdist"] = (1 - same_cbsa) * np.log(move_long["DIST"] + 1)
move_long["destchoice_cbsa_dist"] = same_cbsa * move_long["DIST"]
move_long["destchoice_samecbsa"] = same_cbsa
move_long["destchoice_samestate"] = same_state
move_long["destchoice_birthstate"] = (move_long["POBP"] == move_long["STATE"]).astype(
    float
)
move_long["destchoice_hh_income"] = move_long["HH_MED_INC"]
move_long["destchoice_proportion_struggling"] = move_long["POVERTY_PCT"]
move_long["destchoice_yrs_since_median_structure_built"] = move_long[
    "YR_SINCE_MED_STRUCTURE"
]
move_long["destchoice_median_house_value_over_median_income"] = move_long[
    "MED_HOUSE_VALUE_OVER_MED_HH_INC"
]
move_long["destchoice_median_gross_rent_percentage_hh_inc"] = move_long[
    "MED_RENT_PCT_HH_INC"
]
move_long["destchoice_median_owner_costs"] = move_long["MED_OWNER_COST_HH_INC_PCT"]
move_long["destchoice_unemp_rate"] = move_long["UNEMP_RATE"]
move_long["destchoice_vacancy_rate"] = move_long["HOUSE_VACANCY_PCT"]
move_long["destchoice_proportion_college"] = (
    move_long["IN_COLLEGE"] * move_long["COLLEGE_PCT"]
)
move_long["destchoice_proportion_foreign"] = (
    move_long["FOREIGN"] * move_long["FOREIGN_BORN_PCT"]
)
move_long["destchoice_proportion_same_age_18_34"] = (
    move_long["AGE_18_34"] * move_long["OWN_AGE_PCT"]
)
move_long["destchoice_proportion_same_age_35_64"] = (
    move_long["AGE_35_64"] * move_long["OWN_AGE_PCT"]
)
move_long["destchoice_proportion_same_age_over_65"] = (
    move_long["AGE_OVER_65"] * move_long["OWN_AGE_PCT"]
)
move_long["destchoice_proportion_same_race_black"] = (
    move_long["BLACK"] * move_long["OWN_RACE_PCT"]
)
move_long["destchoice_proportion_same_race_aapi"] = (
    move_long["AAPI"] * move_long["OWN_RACE_PCT"]
)
move_long["destchoice_proportion_same_race_indian"] = (
    move_long["INDIAN"] * move_long["OWN_RACE_PCT"]
)
move_long["destchoice_proportion_same_race_latino"] = (
    move_long["LATINO"] * move_long["OWN_RACE_PCT"]
)
move_long["destchoice_proportion_hh_with_children"] = (
    move_long["HH_WITH_CHILD_PCT"] * move_long["CHILD"]
)
move_long["destchoice_proportion_ent"] = move_long["ENT_JOBS_PCT"]
move_long["destchoice_proportion_ent_18_34"] = (
    move_long["AGE_18_34"] * move_long["ENT_JOBS_PCT"]
)
move_long["destchoice_proportion_ent_35_64"] = (
    move_long["AGE_35_64"] * move_long["ENT_JOBS_PCT"]
)
move_long["destchoice_median_travel_time"] = move_long["MED_TRAVEL_TIME"]
move_long["destchoice_proportion_alt_commute"] = move_long["ALT_COMMUTE_PCT"]
move_long["destchoice_t34_metro"] = same_type_t34 * alt_type_metro
move_long["destchoice_t34_nonmetro"] = same_type_t34 * alt_type_nonmetro
move_long["destchoice_metro_t34"] = same_type_metro * alt_type_t34
move_long["destchoice_metro_metro"] = same_type_metro * alt_type_metro
move_long["destchoice_metro_nonmetro"] = same_type_metro * alt_type_nonmetro
move_long["destchoice_nonmetro_t34"] = same_type_nonmetro * alt_type_t34
move_long["destchoice_nonmetro_metro"] = same_type_nonmetro * alt_type_metro
move_long["destchoice_nonmetro_nonmetro"] = same_type_nonmetro * alt_type_nonmetro
move_long["destchoice_proportion_military"] = (
    move_long["IN_MILITARY"] * move_long["MIL_PCT"]
)
# ALT{i}_OWN_GROUP_PCT is already gathered against each person's own NAICS_{group}
# (see create_estdata.ipynb), so multiplying by each group's dummy just assigns that
# group its own coefficient -- same pattern as OWN_AGE_PCT/OWN_RACE_PCT above.
move_long["destchoice_same_naics_proportion_govt"] = (
    move_long["NAICS_GOVT"] * move_long["OWN_GROUP_PCT"]
)
move_long["destchoice_same_naics_proportion_goods_trade"] = (
    move_long["NAICS_GOODS_TRADE"] * move_long["OWN_GROUP_PCT"]
)
move_long["destchoice_same_naics_proportion_license"] = (
    move_long["NAICS_LICENSE"] * move_long["OWN_GROUP_PCT"]
)
move_long["destchoice_same_naics_proportion_high_ed"] = (
    move_long["NAICS_HIGH_ED"] * move_long["OWN_GROUP_PCT"]
)
move_long["destchoice_same_naics_proportion_agr_ext"] = (
    move_long["NAICS_AGR_EXT"] * move_long["OWN_GROUP_PCT"]
)

MOVE_TERMS = [
    "destchoice_dist",
    "destchoice_logdist",
    "destchoice_cbsa_dist",
    "destchoice_samecbsa",
    "destchoice_samestate",
    "destchoice_birthstate",
    "destchoice_hh_income",
    "destchoice_proportion_struggling",
    "destchoice_yrs_since_median_structure_built",
    "destchoice_median_house_value_over_median_income",
    "destchoice_median_gross_rent_percentage_hh_inc",
    "destchoice_median_owner_costs",
    "destchoice_unemp_rate",
    "destchoice_vacancy_rate",
    "destchoice_proportion_college",
    "destchoice_proportion_foreign",
    "destchoice_proportion_same_age_18_34",
    "destchoice_proportion_same_age_35_64",
    "destchoice_proportion_same_age_over_65",
    "destchoice_proportion_same_race_black",
    "destchoice_proportion_same_race_aapi",
    "destchoice_proportion_same_race_indian",
    "destchoice_proportion_same_race_latino",
    "destchoice_proportion_hh_with_children",
    "destchoice_proportion_ent",
    "destchoice_proportion_ent_18_34",
    "destchoice_proportion_ent_35_64",
    "destchoice_median_travel_time",
    "destchoice_proportion_alt_commute",
    "destchoice_t34_metro",
    "destchoice_t34_nonmetro",
    "destchoice_metro_t34",
    "destchoice_metro_metro",
    "destchoice_metro_nonmetro",
    "destchoice_nonmetro_t34",
    "destchoice_nonmetro_metro",
    "destchoice_nonmetro_nonmetro",
    "destchoice_proportion_military",
    "destchoice_same_naics_proportion_govt",
    "destchoice_same_naics_proportion_goods_trade",
    "destchoice_same_naics_proportion_license",
    "destchoice_same_naics_proportion_high_ed",
    "destchoice_same_naics_proportion_agr_ext",
]
move_long = move_long[["person_id", "alt", "choice", "log_tot_pop"] + MOVE_TERMS]
print(move_long.shape)


### Stay block: one `alt=0` row per person, direct (no `wide_to_long` — different variables entirely, not an alternative-varying reshape)

In [ ]:
stay = df.copy()
# every column here is named to match its corresponding c_stay_* variable in
# modeling_mnl.ipynb, minus the c_ prefix (same convention used for MOVE_TERMS above)
stay["stay"] = 1.0  # c_stay: the stay alternative-specific constant
stay["stay_age_18_22"] = stay["AGE_18_22"]
stay["stay_age_23_29"] = stay["AGE_23_29"]
stay["stay_age_30_39"] = stay["AGE_30_39"]
stay["stay_age_40_49"] = stay["AGE_40_49"]
stay["stay_age_50_64"] = stay["AGE_50_64"]
# stay["stay_age_over_65"] = stay["AGE_OVER_65"]  # reference

stay["stay_proportion_same_age_18_34"] = (
    stay["Proportion of people 18-34.ORIG"] * stay["AGE_18_34"]
)
stay["stay_proportion_same_age_35_64"] = (
    stay["Proportion of people 35-64.ORIG"] * stay["AGE_35_64"]
)
stay["stay_proportion_same_age_65_plus"] = (
    stay["Proportion of people 65+.ORIG"] * stay["AGE_OVER_65"]
)

stay["stay_child_under_6"] = stay["CHILD_UNDER_6"]
stay["stay_child_6_to_17"] = stay["CHILD_6_TO_17"]
stay["stay_proportion_hh_with_children"] = (
    stay["Proportion of households with children.ORIG"] * stay["CHILD"]
)

stay["stay_married_more_than_year"] = stay["MARRIED_MORE_THAN_YEAR"]
stay["stay_married_less_than_year"] = stay["RECENTLY_MARRIED"]
stay["stay_recently_divorced_or_widowed"] = stay["RECENTLY_WIDOWED_OR_DIVORCED"]
stay["stay_2work_mar"] = stay["WORK2_MAR"]
stay["stay_1work_mar"] = stay["WORK1_MAR"]
stay["stay_single_parent"] = stay["SINGLE_PARENT"]

stay["stay_t34"] = (stay["TYPE_NUM.ORIG"] == 0).astype(float)
stay["stay_metro"] = (stay["TYPE_NUM.ORIG"] == 1).astype(float)
# stay["stay_nonmetro"] = (stay["TYPE_NUM.ORIG"] == 2).astype(float) # reference

stay["stay_edu_college"] = stay["EDU_BACHELORS"]
stay["stay_edu_high"] = stay["EDU_HIGH"]
# stay["stay_edu_nohigh"] = stay["EDU_NOHIGH"] # reference

stay["stay_in_college"] = stay["IN_COLLEGE"]
stay["stay_proportion_in_college_if_in_college"] = (
    stay["Proportion of people in college.ORIG"] * stay["IN_COLLEGE"]
)

stay["stay_foreign"] = stay["FOREIGN"]
stay["stay_proportion_foreign_if_foreign"] = (
    stay["Proportion foreign born.ORIG"] * stay["FOREIGN"]
)

stay["stay_proportion_same_race_black"] = (
    stay["Proportion of people Black.ORIG"] * stay["BLACK"]
)
stay["stay_proportion_same_race_aapi"] = (
    stay["Proportion of people AAPI.ORIG"] * stay["AAPI"]
)
stay["stay_proportion_same_race_indian"] = (
    stay["Proportion of people Indian.ORIG"] * stay["INDIAN"]
)
stay["stay_proportion_same_race_latino"] = (
    stay["Proportion of people Latino.ORIG"] * stay["LATINO"]
)

stay["stay_hh_income"] = stay[
    "Median Household Income (In 2018 Inflation Adjusted Dollars).Median Household Income (In 2018 Inflation Adjusted Dollars).SE_A14006_001.ORIG"
]
stay["stay_proportion_struggling"] = stay["Proportion of people struggling.ORIG"]
stay["stay_yrs_since_median_structure_built"] = stay[
    "Years since median structure built.ORIG"
]
stay["stay_median_house_value_over_median_income"] = stay[
    "Median house value over median household income.ORIG"
]
stay["stay_median_gross_rent_percentage_hh_inc"] = stay[
    "Median gross rent as a percentage of household income.ORIG"
]
stay["stay_median_owner_costs"] = stay[
    "Median selected monthly owner costs as percentage of household income.ORIG"
]

stay["stay_unemp_rate"] = stay["Unemployment rate.ORIG"]
stay["stay_vacancy_rate"] = stay["House vacancy proportion.ORIG"]
stay["stay_median_travel_time"] = stay["Median travel time.ORIG"]
stay["stay_proportion_alt_commute"] = stay["Proportion alternative commute.ORIG"]

stay["stay_mil"] = stay["IN_MILITARY"]
stay["stay_naics_govt"] = stay["NAICS_GOVT"]
stay["stay_naics_goods_trade"] = stay["NAICS_GOODS_TRADE"]
stay["stay_naics_license"] = stay["NAICS_LICENSE"]
stay["stay_naics_high_ed"] = stay["NAICS_HIGH_ED"]
stay["stay_naics_agr_ext"] = stay["NAICS_AGR_EXT"]

stay["stay_proportion_mil"] = (
    stay["Proportion of people in military.ORIG"] * stay["IN_MILITARY"]
)
stay["stay_naics_proportion_govt"] = (
    stay["NAICS_GROUP_PCT_GOVT.ORIG"] * stay["NAICS_GOVT"]
)
stay["stay_naics_proportion_goods_trade"] = (
    stay["NAICS_GROUP_PCT_GOODS_TRADE.ORIG"] * stay["NAICS_GOODS_TRADE"]
)
stay["stay_naics_proportion_license"] = (
    stay["NAICS_GROUP_PCT_LICENSE.ORIG"] * stay["NAICS_LICENSE"]
)
stay["stay_naics_proportion_high_ed"] = (
    stay["NAICS_GROUP_PCT_HIGH_ED.ORIG"] * stay["NAICS_HIGH_ED"]
)
stay["stay_naics_proportion_agr_ext"] = (
    stay["NAICS_GROUP_PCT_AGR_EXT.ORIG"] * stay["NAICS_AGR_EXT"]
)

STAY_TERMS = [
    "stay",
    "stay_age_18_22",
    "stay_age_23_29",
    "stay_age_30_39",
    "stay_age_40_49",
    "stay_age_50_64",
    "stay_proportion_same_age_18_34",
    "stay_proportion_same_age_35_64",
    "stay_proportion_same_age_65_plus",
    "stay_child_under_6",
    "stay_child_6_to_17",
    "stay_proportion_hh_with_children",
    "stay_married_more_than_year",
    "stay_married_less_than_year",
    "stay_recently_divorced_or_widowed",
    "stay_2work_mar",
    "stay_1work_mar",
    "stay_single_parent",
    "stay_t34",
    "stay_metro",
    "stay_edu_college",
    "stay_edu_high",
    "stay_in_college",
    "stay_proportion_in_college_if_in_college",
    "stay_foreign",
    "stay_proportion_foreign_if_foreign",
    "stay_proportion_same_race_black",
    "stay_proportion_same_race_aapi",
    "stay_proportion_same_race_indian",
    "stay_proportion_same_race_latino",
    "stay_hh_income",
    "stay_proportion_struggling",
    "stay_yrs_since_median_structure_built",
    "stay_median_house_value_over_median_income",
    "stay_median_gross_rent_percentage_hh_inc",
    "stay_median_owner_costs",
    "stay_unemp_rate",
    "stay_vacancy_rate",
    "stay_median_travel_time",
    "stay_proportion_alt_commute",
    "stay_mil",
    "stay_naics_govt",
    "stay_naics_goods_trade",
    "stay_naics_license",
    "stay_naics_high_ed",
    "stay_naics_agr_ext",
    "stay_proportion_mil",
    "stay_naics_proportion_govt",
    "stay_naics_proportion_goods_trade",
    "stay_naics_proportion_license",
    "stay_naics_proportion_high_ed",
    "stay_naics_proportion_agr_ext",
]
stay["alt"] = 0
stay["choice"] = (stay["STAY"] == 1).astype(int)
stay = stay[["person_id", "alt", "choice"] + STAY_TERMS]
print(stay.shape)


### Concatenate and fit

Each block only has its own terms; the other block's terms are `0` on its rows (structurally zero, not missing). `fit_intercept=False` since `stay` (the c_stay analog) is already an explicit ASC for staying. `log_tot_pop` is `addit` (fixed offset, coefficient pinned to 1), matching the unweighted `log(ALT{i}_TOT_POP)` term in the biogeme spec.

In [ ]:
long_df = pd.concat([stay, move_long], ignore_index=True, sort=False)
varnames = STAY_TERMS + MOVE_TERMS
long_df[varnames + ["log_tot_pop"]] = long_df[varnames + ["log_tot_pop"]].fillna(0.0)
long_df = long_df.sort_values(["person_id", "alt"]).reset_index(drop=True)

model = MultinomialLogit()
model.fit(
    X=long_df[varnames],
    y=long_df["choice"],
    varnames=varnames,
    ids=long_df["person_id"],
    alts=long_df["alt"],
    addit=long_df["log_tot_pop"],
    fit_intercept=False,
)
model.summary()
